# Model_Comparison
**Project:** Hotel Booking Demand - Supervised Learning Assignment  
**Purpose:** Load the four saved model files, evaluate them on the same test split, compare them across metrics, and identify the best-performing model.

## 1. Imports

In [ ]:
# ============================================================
# Cell 1: Import comparison libraries and helper functions
# ============================================================
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

from hotel_booking_common import (
    RANDOM_STATE,
    get_project_root,
    ensure_dir,
    load_dataset,
    prepare_hotel_booking_dataframe,
    build_train_test_split,
    compare_models_on_test_set,
)

## 2. Project configuration

In [ ]:
# ============================================================
# Cell 2: Set project paths for input data and comparison outputs
# ============================================================
PROJECT_ROOT = get_project_root()
DATASET_DIR = PROJECT_ROOT / "dataset"
OUTPUT_DIR = ensure_dir(PROJECT_ROOT / "outputs" / "comparison")

MODEL_PATHS = {
    "Logistic Regression": PROJECT_ROOT / "outputs" / "logistic_regression" / "best_model.joblib",
    "Decision Tree": PROJECT_ROOT / "outputs" / "decision_tree" / "best_model.joblib",
    "Random Forest": PROJECT_ROOT / "outputs" / "random_forest" / "best_model.joblib",
    "Gradient Boosting": PROJECT_ROOT / "outputs" / "gradient_boosting" / "best_model.joblib",
}

for model_name, model_path in MODEL_PATHS.items():
    print(model_name, "->", model_path)

## 3. Load and preprocess dataset

In [ ]:
# ============================================================
# Cell 3: Recreate the same cleaned dataset and hold-out split
# ============================================================
df_raw, dataset_path = load_dataset(DATASET_DIR)
df_model, preprocessing_notes = prepare_hotel_booking_dataframe(df_raw, target_col="is_canceled")

X_train, X_test, y_train, y_test = build_train_test_split(
    df_model,
    target_col="is_canceled",
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"Comparison test rows: {X_test.shape[0]}")

## 4. Compare the saved models

In [ ]:
# ============================================================
# Cell 4: Load the saved pipelines and compare them consistently
# ============================================================
missing_models = [name for name, path in MODEL_PATHS.items() if not path.exists()]
if missing_models:
    raise FileNotFoundError(
        "Run all four algorithm notebooks first. Missing model files for: " + ", ".join(missing_models)
    )

comparison_df = compare_models_on_test_set(
    model_paths=MODEL_PATHS,
    X_test=X_test,
    y_test=y_test,
    output_dir=OUTPUT_DIR,
)

display(comparison_df)

## 5. Identify the best model

In [ ]:
# ============================================================
# Cell 5: Extract and display the best overall model from the comparison
# ============================================================
best_model_row = comparison_df.iloc[0]
display(Markdown("### Best model based on ROC-AUC, F1, and MCC ranking"))
display(best_model_row.to_frame("value"))

## 6. Save a short comparison report

In [ ]:
# ============================================================
# Cell 6: Create a short markdown report for the final write-up
# ============================================================
best_model_name = comparison_df.iloc[0]["model_name"]
report_lines = [
    "# Model Comparison Report",
    "",
    f"- Dataset file: {dataset_path.name}",
    f"- Test set size: {X_test.shape[0]}",
    f"- Best overall model: {best_model_name}",
    "",
    "## Ranking table",
    comparison_df.to_markdown(index=False),
    "",
    "## Discussion prompts",
    "- Compare performance trade-offs between interpretability and accuracy.",
    "- Discuss whether the best ROC-AUC model is also the best MCC or recall model.",
    "- Use the saved plots in outputs/comparison to support the report discussion."
]
report_path = OUTPUT_DIR / "model_comparison_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print(f"Saved comparison report to: {report_path}")